# Variational data assimilation with diffWOFOST

The overall goal of data assimilation is to combine all possible information, including models, observations, a-priori knowledge, and uncertainty estimates, in order to obtain the best possible estimate of the state of a system.

Variational data assimilation does this by posing one estimation problem over a time window. Instead of correcting the simulation each time a measurement arrives, it searches for a set of uncertain model inputs that makes the simulated trajectory agree as well as possible with the observations while remaining close to prior knowledge.

In practice this means defining a cost function, running the model, measuring the mismatch between model and observations, and adjusting parameters or other controllable quantities until that mismatch is reduced. In crop modelling this is attractive because observational information influences the full seasonal trajectory in a physically consistent way.

This notebook focuses on that variational route using diffWOFOST, the differentiable implementation of WOFOST in PyTorch. Because gradients can be propagated through the simulation, gradient-based optimization can be used directly to solve the assimilation problem.

Earlier notebooks in this series demonstrated the Kalman-filter route, in particular Ensemble Kalman Filter workflows for sequentially updating model states from LAI and soil-moisture observations. Here we take the complementary approach: rather than performing sequential analysis updates, we optimize a differentiable objective over the full assimilation window using the same synthetic observations.

## 1. Background on data assimilation

### 1.1 Sequential versus variational data assimilation

Data-assimilation methods are often grouped into two broad families.

1. **Sequential methods** run the model forward in time until an observation becomes available. At that point an analysis step is carried out that adjusts the model state using the observation and its uncertainty, after which the model is integrated forward again until the next observation arrives.

2. **Variational methods** collect all observations within an assimilation window and then fit the model to those observations by minimizing a cost function. In that optimization, one can adjust parameters, initial conditions, or other attributes that control the simulated trajectory.

A compact formulation is:

$$

J(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left(\frac{y_i^{\mathrm{model}}(\theta) - y_i^{\mathrm{obs}}}{\sigma_i}\right)^2 + \lambda \frac{1}{P} \sum_{j=1}^{P} \left(\frac{\theta_j - \theta_{j,b}}{s_j}\right)^2

$$

where:

- $y_i^{\mathrm{obs}}$ are the observations at the measurement dates,
- $\sigma_i$ are the observation uncertainties,
- $\theta_{j,b}$ are prior or background parameter values,
- $s_j$ are scales used to normalize departures from the prior,
- and $\lambda$ controls the strength of the background penalty.

The first term rewards agreement with observations. The second term discourages unrealistic departures from prior knowledge. Methods such as 3D-Var and 4D-Var differ in how the state and time window are defined, but the central idea is the same: optimize a full trajectory rather than applying isolated corrections.

The earlier `pcse` notebooks illustrate the sequential viewpoint using [Ensemble Kalman Filter (EnKF)](https://github.com/ajwdewit/pcse_notebooks/blob/master/08a%20Data%20assimilation%20with%20the%20EnKF.ipynb) and [EnKF Multistate](https://github.com/ajwdewit/pcse_notebooks/blob/master/08b%20Data%20assimilation%20with%20the%20EnKF%20multistate.ipynb). There, the simulated state is propagated forward until an observation is available, then updated through an analysis step that can introduce a jump in the state trajectory. That Kalman-style workflow is not used here, but it provides a useful contrast for understanding what is specific about the variational approach.

One practical consequence is worth keeping in mind: sequential state updates are inherently non-conservative, because mass or energy can be added or removed when the state is corrected. In a crop model this means that carbon, water, and nutrient balances may no longer close automatically. Variational methods avoid those explicit state jumps and instead search for a trajectory that is generated consistently by the model itself.

### 1.2 Why diffWOFOST is useful here

The main practical difficulty in variational assimilation is optimization. To minimize $J(\theta)$ efficiently, one usually needs gradients of the model outputs with respect to the uncertain inputs. For conventional crop-model implementations, those gradients can be hard to derive or expensive to approximate numerically.

`diffWOFOST` addresses that by implementing the model in PyTorch. As a result:

- model parameters can be treated as trainable variables,
- gradients can flow through the simulation,
- and standard optimizers such as Adam can be used to fit the model to observations.

### 1.3 Scope of this notebook

This notebook uses a synthetic observation record consisting of five dates of leaf area index (`LAI`) and soil moisture (`SM`), matching the earlier Kalman-filter examples so that the difference lies in the assimilation method rather than in the data.

The goal is not to reproduce an operational assimilation system, but to demonstrate the variational workflow clearly:

1. start from a prior simulation,
2. compare it with observations,
3. define a differentiable loss function,
4. optimize a small set of parameters,
5. inspect how the analysed trajectory changes.

## 2. Applying variational data assimilation with diffWOFOST




This section builds the workflow step by step: import the model components, define the observations, run a prior simulation, and then convert that simulation into a variational optimization problem.

In [4]:
%matplotlib inline

from pathlib import Path
import copy
import datetime as dt

import matplotlib
matplotlib.style.use("ggplot")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalanceFD
from diffwofost.physical_models.utils import get_test_data, prepare_engine_input

ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

print(f"torch version: {torch.__version__}")
print(f"device: {ComputeConfig.get_device()}")
print(f"dtype: {ComputeConfig.get_dtype()}")

torch version: 2.9.0+cu128
device: cpu
dtype: torch.float64


### 2.2 The model, the observations, and a baseline run




We use a local diffWOFOST regression dataset for a winter-wheat water-limited run together with a synthetic observation schedule for `LAI` and `SM`. The observations are written out explicitly so that every element of the example is visible in the notebook itself.



The workflow is deliberately simple:



1. define the model inputs,

2. define the observations and their uncertainty,

3. run a baseline simulation without assimilation,

4. compare the baseline trajectory with the observations before optimization.

#### 2.2.1 Defining the synthetic observations




As in the EnKF examples, we prescribe five observation dates for leaf area index (`LAI`) and soil moisture (`SM`) together with standard deviations that represent observation uncertainty.

In [5]:
test_data_path = Path("../tests/physical_models/test_data/test_waterlimitedproduction_wofost72_03.yaml")
test_data = get_test_data(test_data_path)

crop_model_params = [
    # Leaf dynamics
    "SPAN",
    "TDWI",
    "TBASE",
    "PERDL",
    "RGRLAI",
    "KDIFTB",
    "SLATB",
    # Phenology
    "TSUMEM",
    "TBASEM",
    "TEFFMX",
    "TSUM1",
    "TSUM2",
    "DLO",
    "DLC",
    "DVSI",
    "DVSEND",
    "DTSMTB",
    # Assimilation
    "AMAXTB",
    "EFFTB",
    "TMPFTB",
    "TMNFTB",
    # Respiration
    "Q10",
    "RMR",
    "RML",
    "RMS",
    "RMO",
    "RFSETB",
    # Evapotranspiration and soil water
    "CFET",
    "DEPNR",
    "IAIRDU",
    "IOX",
    "CRAIRC",
    "SM0",
    "SMW",
    "SMFCF",
    # Root dynamics
    "RDI",
    "RRI",
    "RDMCR",
    "RDMSOL",
    "RDRRTB",
    # Stem dynamics
    "RDRSTB",
    "SSATB",
    # Storage organs
    "SPA",
    # Partitioning
    "FRTB",
    "FLTB",
    "FSTB",
    "FOTB",
    # Conversion factors
    "CVL",
    "CVO",
    "CVR",
    "CVS",
]

dates_of_observation = [
    dt.date(2000, 2, 7),
    dt.date(2000, 2, 28),
    dt.date(2000, 3, 20),
    dt.date(2000, 4, 10),
    dt.date(2000, 5, 1),
]
observed_lai = np.array([2.2, 3.5, 6.2, 3.3, 2.1], dtype=float)
std_lai = observed_lai * 0.10
observed_sm = np.array([0.285, 0.26, 0.28, 0.18, 0.17], dtype=float)
std_sm = observed_sm * 0.05

observation_df = pd.DataFrame({
    "day": dates_of_observation,
    "LAI_obs": observed_lai,
    "LAI_std": std_lai,
    "SM_obs": observed_sm,
    "SM_std": std_sm,
}).set_index("day")
observation_df

,LAI_obs,LAI_std,SM_obs,SM_std
day,,,,
2000-02-07,2.2,0.22,0.285,0.01425
2000-02-28,3.5,0.35,0.260,0.01300
2000-03-20,6.2,0.62,0.280,0.01400
2000-04-10,3.3,0.33,0.180,0.00900
2000-05-01,2.1,0.21,0.170,0.00850


#### 2.2.2 A standard diffWOFOST run




Before assimilating anything, we run the model once with the prior parameter set. This baseline trajectory plays the same role as the open-loop simulation in the original notebook.

In [6]:
wofost72_fd_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalanceFD,
    OUTPUT_VARS=[
        "DVS",
        "LAI",
        "SM",
        "TAGP",
        "TRA",
        "TWLV",
        "TWRT",
        "TWSO",
        "TWST",
    ],
)

def load_engine_inputs():
    return prepare_engine_input(test_data, crop_model_params)

def scalarize(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu())
    return value

def results_to_frame(results):
    rows = []
    for row in results:
        rows.append({key: (value if key == "day" else scalarize(value)) for key, value in row.items()})
    return pd.DataFrame(rows).set_index("day")

provider, weather, agromanagement, external_states = load_engine_inputs()
baseline_engine = Engine(config=wofost72_fd_config)
baseline_engine.setup(provider, weather, agromanagement, external_states)
baseline_engine.run_till_terminate()
baseline_results = baseline_engine.get_output()
baseline_df = results_to_frame(baseline_results)

baseline_df.loc[dates_of_observation, ["LAI", "SM", "TWST", "TWSO", "TAGP"]]

KeyError: "None of [Index([2000-02-07, 2000-02-28, 2000-03-20, 2000-04-10, 2000-05-01], dtype='object', name='day')] are in the [index]"

#### 2.2.3 Plotting the baseline model against the observations




This figure is the variational counterpart of the first comparison plot in the original notebook: it shows how the prior simulation differs from the observations before any optimization is applied.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), sharex=True)
baseline_df["LAI"].plot(ax=axes[0], label="baseline model")
axes[0].errorbar(dates_of_observation, observed_lai, yerr=std_lai, fmt="o", label="synthetic observations")
axes[0].set_title("LAI: baseline vs observations")
axes[0].legend()

baseline_df["SM"].plot(ax=axes[1], label="baseline model")
axes[1].errorbar(dates_of_observation, observed_sm, yerr=std_sm, fmt="o", label="synthetic observations")
axes[1].set_title("Root-zone soil moisture: baseline vs observations")
axes[1].legend()

fig.autofmt_xdate()
plt.show()

### 2.3 Setting up the variational problem




The next step mirrors the role of the analysis step in the EnKF notebooks, but the mechanism is different. Instead of updating state variables directly, we define a small set of trainable parameters and optimize them against the observations over the full window.



We optimize four bounded parameters:



- `TDWI`: total initial dry weight,

- `SPAN`: leaf life span,

- `DVSI`: initial development stage,

- `SMFCF`: soil moisture at field capacity.



The cost function contains:



- an observation term for `LAI`,

- an observation term for `SM`,

- and a weak background penalty that keeps the optimized values close to the prior parameter values from the YAML setup.

In [ ]:
obs_lai_t = torch.tensor(observed_lai, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
obs_lai_std_t = torch.tensor(std_lai, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
obs_sm_t = torch.tensor(observed_sm, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
obs_sm_std_t = torch.tensor(std_sm, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())

class BoundedParameter(torch.nn.Module):
    def __init__(self, low, high, init_value):
        super().__init__()
        self.low = low
        self.high = high
        init_norm = (init_value - low) / (high - low)
        init_norm = min(max(init_norm, 1e-6), 1 - 1e-6)
        self.raw = torch.nn.Parameter(
            torch.logit(
                torch.tensor(init_norm, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device()),
                eps=1e-6,
            )
        )

    def forward(self):
        return self.low + (self.high - self.low) * torch.sigmoid(self.raw)

class VariationalDiffWofost72(torch.nn.Module):
    def __init__(self, crop_model_params_provider, weather_data_provider, agro_management_inputs, external_states, config):
        super().__init__()
        self.base_provider = crop_model_params_provider
        self.weather_data_provider = weather_data_provider
        self.agro_management_inputs = agro_management_inputs
        self.external_states = external_states
        self.config = config
        self.engine = Engine(config=config)

        self.tdwi = BoundedParameter(0.10, 1.20, init_value=0.35)
        self.span = BoundedParameter(10.0, 60.0, init_value=28.0)
        self.dvsi = BoundedParameter(-0.10, 0.50, init_value=0.10)
        self.smfcf = BoundedParameter(0.18, 0.45, init_value=0.26)

    def parameter_dict(self):
        return {
            "TDWI": self.tdwi(),
            "SPAN": self.span(),
            "DVSI": self.dvsi(),
            "SMFCF": self.smfcf(),
        }

    def run_model(self, overrides):
        provider = copy.deepcopy(self.base_provider)
        for name, value in overrides.items():
            provider.set_override(name, value, check=False)

        engine = self.engine.setup(
            provider,
            self.weather_data_provider,
            self.agro_management_inputs,
            self.external_states,
        )
        engine.run_till_terminate()
        return engine.get_output()

    def forward(self):
        overrides = self.parameter_dict()
        results = self.run_model(overrides)
        by_day = {row["day"]: row for row in results}
        lai_at_obs = torch.stack([by_day[day]["LAI"] for day in dates_of_observation])
        sm_at_obs = torch.stack([by_day[day]["SM"] for day in dates_of_observation])
        return lai_at_obs, sm_at_obs, overrides

provider_opt, weather_opt, agro_opt, external_opt = load_engine_inputs()
prior_params = {
    "TDWI": provider_opt["TDWI"].detach().clone(),
    "SPAN": provider_opt["SPAN"].detach().clone(),
    "DVSI": provider_opt["DVSI"].detach().clone(),
    "SMFCF": provider_opt["SMFCF"].detach().clone(),
}
background_scales = {
    "TDWI": torch.tensor(0.15, dtype=ComputeConfig.get_dtype()),
    "SPAN": torch.tensor(8.0, dtype=ComputeConfig.get_dtype()),
    "DVSI": torch.tensor(0.20, dtype=ComputeConfig.get_dtype()),
    "SMFCF": torch.tensor(0.05, dtype=ComputeConfig.get_dtype()),
}

opt_model = VariationalDiffWofost72(
    provider_opt,
    weather_opt,
    agro_opt,
    external_opt,
    wofost72_fd_config,
)

{name: float(value) for name, value in prior_params.items()}

In [ ]:
optimizer = torch.optim.Adam(opt_model.parameters(), lr=0.03)
history = []
best_state = None
best_loss = float("inf")
background_weight = 0.05

for step in range(121):
    optimizer.zero_grad()
    lai_pred, sm_pred, params = opt_model()

    lai_loss = torch.mean(((lai_pred - obs_lai_t) / obs_lai_std_t) ** 2)
    sm_loss = torch.mean(((sm_pred - obs_sm_t) / obs_sm_std_t) ** 2)
    background_loss = torch.mean(
        torch.stack([
            ((params[name] - prior_params[name]) / background_scales[name]) ** 2
            for name in params
        ])
    )
    loss = lai_loss + sm_loss + background_weight * background_loss
    loss.backward()
    optimizer.step()

    snapshot = {name: float(value.detach().cpu()) for name, value in params.items()}
    history.append({
        "step": step,
        "loss": float(loss.detach().cpu()),
        "lai_loss": float(lai_loss.detach().cpu()),
        "sm_loss": float(sm_loss.detach().cpu()),
        "background_loss": float(background_loss.detach().cpu()),
        **snapshot,
    })

    if history[-1]["loss"] < best_loss:
        best_loss = history[-1]["loss"]
        best_state = {name: value.detach().clone() for name, value in params.items()}

    if step % 20 == 0 or step == 120:
        print(
            f"step={step:03d} loss={history[-1]['loss']:.4f} "
            f"lai={history[-1]['lai_loss']:.4f} sm={history[-1]['sm_loss']:.4f} "
            f"TDWI={snapshot['TDWI']:.4f} SPAN={snapshot['SPAN']:.4f} "
            f"DVSI={snapshot['DVSI']:.4f} SMFCF={snapshot['SMFCF']:.4f}"
        )

history_df = pd.DataFrame(history)
history_df.tail()

In [ ]:
with torch.no_grad():
    analysed_results = opt_model.run_model(best_state)

analysis_df = results_to_frame(analysed_results)
comparison_at_obs = pd.DataFrame({
    "LAI_obs": observed_lai,
    "LAI_baseline": baseline_df.loc[dates_of_observation, "LAI"].to_numpy(),
    "LAI_analysis": analysis_df.loc[dates_of_observation, "LAI"].to_numpy(),
    "SM_obs": observed_sm,
    "SM_baseline": baseline_df.loc[dates_of_observation, "SM"].to_numpy(),
    "SM_analysis": analysis_df.loc[dates_of_observation, "SM"].to_numpy(),
}, index=dates_of_observation)

print("Best parameter values")
for name, value in best_state.items():
    print(f"{name}: {float(value):.4f} (prior {float(prior_params[name]):.4f})")

comparison_at_obs

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10), sharex=True)

history_df.plot(x="step", y=["loss", "lai_loss", "sm_loss"], ax=axes[0, 0])
axes[0, 0].set_title("Loss history")

baseline_df["LAI"].plot(ax=axes[0, 1], label="baseline")
analysis_df["LAI"].plot(ax=axes[0, 1], label="variational analysis")
axes[0, 1].errorbar(dates_of_observation, observed_lai, yerr=std_lai, fmt="o", label="observed LAI")
axes[0, 1].set_title("LAI fit over the assimilation window")
axes[0, 1].legend()

baseline_df["SM"].plot(ax=axes[1, 0], label="baseline")
analysis_df["SM"].plot(ax=axes[1, 0], label="variational analysis")
axes[1, 0].errorbar(dates_of_observation, observed_sm, yerr=std_sm, fmt="o", label="observed SM")
axes[1, 0].set_title("SM fit over the assimilation window")
axes[1, 0].legend()

baseline_df[["TWST", "TWSO", "TAGP"]].plot(ax=axes[1, 1], style=["--", "--", "--"], alpha=0.9)
analysis_df[["TWST", "TWSO", "TAGP"]].plot(ax=axes[1, 1], style=["-", "-", "-"])
axes[1, 1].set_title("Indirect effects on WST, WSO and TAGP")
axes[1, 1].legend([
    "TWST baseline", "TWSO baseline", "TAGP baseline",
    "TWST analysis", "TWSO analysis", "TAGP analysis",
])

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Visualizing and interpreting the variational analysis




The optimized trajectory can be interpreted through a few key points:



1. **The analysed trajectory is still a model trajectory.** The simulation is not patched locally at the observation times. Instead, the model is rerun with a better parameter set.

2. **Multiple variables can be fitted together.** Here `LAI` and `SM` appear in the same cost function, so the optimizer balances both observational constraints simultaneously.

3. **Unobserved variables can still respond.** Quantities such as `WST`, `WSO`, and `TAGP` are not observed directly in the loss, but they change because the optimized parameters alter the full crop trajectory.

4. **Regularization matters.** The background term prevents the optimizer from drifting too far from the prior parameter values, which is important when observations are sparse or noisy.



Natural extensions are to widen the assimilation window, optimize more parameters, introduce parameter covariances, or replace the simple weighted least-squares objective with a richer statistical formulation.